# AgriVision — Training Ulang Model Tomat (v2)
### Fokus tomat saja · split bersih per daun · data non-lab (PlantDoc) · uji lapangan terpisah

Notebook ini menggantikan `agrivision_training.ipynb` (v1, tomat + cabai). Cabai sudah dihapus dari proyek.

**Kenapa v2 (bukan sekadar menambah gambar):** angka v1 (akurasi validasi 94,33%) hanya berlaku untuk foto lab. Dataset Kaggle "New Plant Diseases" adalah augmentasi *offline* PlantVillage yang di-split sesudah augmentasi, jadi set validasinya kemungkinan berisi turunan daun yang sama dengan set latih, dan set itu juga dipakai untuk early stopping. Menambah gambar PlantVillage tidak memperbaiki masalah ini — yang membantu adalah (1) split bersih per daun, (2) data berfoto non-lab ikut dilatih, (3) set uji lapangan yang **tidak** ikut dilatih, (4) augmentasi domain.

**Data (keduanya lewat Hugging Face `datasets`, tanpa API key):**
- **PlantVillage (lab):** `mohanty/PlantVillage` (RGB asli, non-augmentasi; `data.zip` + `splits/color_*.txt` + `leaf-map.json` dibaca langsung karena dataset berbasis skrip tidak bisa dimuat dengan `load_dataset`). Split 80/20 bawaan menjaga daun yang sama tidak muncul di kedua sisi. Lisensi CC BY-SA 3.0. Hanya kelas tomat (10 kelas) yang dipakai.
- **PlantDoc (non-lab / "lapangan"):** `Project-AgML/plant_doc_classification` — foto daun dari web/lapangan berlatar beragam, 28 kelas; hanya 8 kelas tomat yang bisa dipetakan ke 10 label kita. Lisensi CC BY-SA 4.0. **Target Spot tidak ada** di PlantDoc dan **spider mites hanya 2 gambar** (dikecualikan), jadi kedua kelas itu tidak punya uji lapangan — laporkan apa adanya.

**Pembagian data:**
| Set | Isi | Dipakai untuk |
|---|---|---|
| `lab_train` | PlantVillage train (dikurangi `lab_val`) | latih |
| `lab_val` | ±10% daun dari train PlantVillage (dipisah per `leaf_id`) | early stopping + pilih model |
| `lab_test` | PlantVillage test (per daun) | **uji lab** (dilaporkan) |
| `field_train` | PlantDoc ±52% | latih (di-oversample) |
| `field_val` | PlantDoc ±18% | early stopping + pilih model |
| `field_test` | PlantDoc ±30% | **uji lapangan** (dilaporkan; tidak ikut latih) |

Model dipilih berdasarkan **akurasi `field_val`** (bukan `field_test`, supaya set uji tetap bersih).

**Sebelum mulai:**
1. Buka di https://colab.research.google.com → Runtime → Change runtime type → **T4 GPU**.
2. (Opsional tapi disarankan) siapkan `ml-service/models/tomato_classifier.tflite` v1 untuk dibandingkan dengan v2 pada set uji yang sama — akan diminta di Step 8.
3. Klik **Runtime → Run all**. Tidak ada kredensial yang dibutuhkan.

**Sitasi wajib:** Mohanty, Hughes & Salathé (2016) untuk PlantVillage; Singh dkk. (2020) untuk PlantDoc. Cantumkan versi/mirror yang dipakai (dicetak di Step 2–3).

In [ ]:
# Cek GPU aktif — kalau kosong/error: Runtime > Change runtime type > T4 GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 0 — Install & import

In [ ]:
!pip install -q datasets scikit-learn seaborn
import os, re, json, math, shutil, random, hashlib
import numpy as np
import tensorflow as tf
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices("GPU"))

## Step 1 — Kelas kanonik (10 kelas tomat)

Urutan **harus identik** dengan v1 / `ml-service/models/label_map.json` (indeks 0–9), supaya `disease_reference` dan service tidak perlu berubah.

In [ ]:
TOMATO_CLASSES = [
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
]
assert TOMATO_CLASSES == sorted(TOMATO_CLASSES), "urutan harus alfabetis (sama dengan v1)"

def norm(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

CANON_BY_NORM = {norm(c): c for c in TOMATO_CLASSES}

DATA_DIR = "/content/data_v2"
SPLITS = ["lab_train", "lab_val", "lab_test", "field_train", "field_val", "field_test"]
shutil.rmtree(DATA_DIR, ignore_errors=True)
for sp in SPLITS:
    for c in TOMATO_CLASSES:
        os.makedirs(f"{DATA_DIR}/{sp}/{c}", exist_ok=True)

# registry: split -> list of (path, class_name, dhash)
records = {sp: [] for sp in SPLITS}

def dhash(img, size=8):
    g = img.convert("L").resize((size + 1, size), Image.LANCZOS)
    a = np.asarray(g, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

def save_image(img, split, cls, idx):
    path = f"{DATA_DIR}/{split}/{cls}/{split}_{idx:06d}.jpg"
    img.convert("RGB").save(path, quality=95)
    return path

## Step 2 — PlantVillage (lab) → `lab_train` / `lab_val` / `lab_test`

Split asli dataset (per daun) dipakai untuk `lab_test`. `lab_val` diambil dari train **dengan `GroupShuffleSplit` per `leaf_id`**, lalu dicek tidak ada `leaf_id` yang bocor antar set.

In [ ]:
# mohanty/PlantVillage adalah dataset berbasis skrip (plant_village.py) yang TIDAK bisa dimuat dengan
# load_dataset() di versi `datasets` terbaru (BuilderConfig 'color' not found). Jadi file mentahnya dibaca
# langsung: data.zip (raw/color/<kelas>/<file>), splits/color_{train,test}.txt, leaf_grouping/leaf-map.json.
from huggingface_hub import hf_hub_download
import zipfile
from sklearn.model_selection import GroupShuffleSplit

REPO = "mohanty/PlantVillage"
zip_path = hf_hub_download(REPO, "data.zip", repo_type="dataset")   # +-2,2 GB
leaf_map = json.load(open(hf_hub_download(REPO, "leaf_grouping/leaf-map.json", repo_type="dataset"), encoding="utf-8"))
split_lines = {
    s: open(hf_hub_download(REPO, f"splits/color_{s}.txt", repo_type="dataset"), encoding="utf-8").read().splitlines()
    for s in ["train", "test"]
}
print("contoh baris split:", split_lines["train"][:2])

zf = zipfile.ZipFile(zip_path)
member_by_rel = {}
for n in zf.namelist():
    if "raw/color/Tomato" in n and not n.endswith("/"):
        member_by_rel[n[n.index("raw/color/"):]] = n
print("file tomat berwarna di zip:", len(member_by_rel), "| contoh:", list(member_by_rel)[:2])
assert member_by_rel, "tidak ada raw/color/Tomato* di data.zip — cek struktur: zf.namelist()[:10]"

def leaf_id_for(filename, class_name):
    # meniru turunan leaf_id di plant_village.py (hasil kira-kira; lihat pengecekan kebocoran di bawah)
    name = filename.replace("_final_masked", "")
    if "___" in name:
        name = name.split("___")[-1]
    low = name.lower()
    if "copy" in low:
        name = name[: low.index("copy")]
    for ext in (".jpg", ".png"):
        if name.lower().endswith(ext):
            name = name[: -len(ext)]
    ident = name.strip().lower()
    sugg = leaf_map.get(ident)
    if isinstance(sugg, str):
        sugg = [sugg]
    if sugg:
        if len(sugg) == 1:
            return str(sugg[0])
        for sg in sugg:
            if class_name in str(sg):
                return str(sg)
    return f"fallback_{ident}"

def parse(lines):
    out = []
    for ln in lines:
        parts = ln.strip().split("/")
        if len(parts) < 4 or parts[1] != "color":
            continue
        if norm(parts[2]) not in CANON_BY_NORM:
            continue
        rel = "/".join(parts)
        if rel in member_by_rel:
            out.append((rel, CANON_BY_NORM[norm(parts[2])], leaf_id_for(parts[-1], parts[2])))
    return out

pv_train_items, pv_test_items = parse(split_lines["train"]), parse(split_lines["test"])
print("PlantVillage tomat — train:", len(pv_train_items), "test:", len(pv_test_items))
assert pv_train_items and pv_test_items, "parse() menghasilkan 0 baris — bandingkan contoh baris split dengan format yang diharapkan (raw/color/<kelas>/<file>)"
n_fb = sum(1 for it in pv_train_items + pv_test_items if it[2].startswith("fallback_"))
print(f"leaf_id fallback (tidak ada di leaf-map): {n_fb} dari {len(pv_train_items) + len(pv_test_items)}")

In [ ]:
# Pengecekan kebocoran per daun: buang dari train semua gambar yang leaf_id-nya juga ada di test
test_leaves = {it[2] for it in pv_test_items}
before = len(pv_train_items)
pv_train_items = [it for it in pv_train_items if it[2] not in test_leaves]
print("gambar train dibuang karena leaf_id ada di test:", before - len(pv_train_items))

# lab_val dipisah per leaf_id dari train
leaf_train = np.array([it[2] for it in pv_train_items])
gss = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)
tr_idx, va_idx = next(gss.split(np.zeros(len(leaf_train)), groups=leaf_train))
assert not (set(leaf_train[tr_idx]) & set(leaf_train[va_idx])), "ada leaf_id yang sama di train dan val"
assert not (set(leaf_train) & test_leaves), "ada leaf_id yang sama di train dan test"
print("OK — tidak ada leaf_id yang sama antara lab_train, lab_val, dan lab_test")

def dump(items, indices, split):
    for i, k in enumerate(indices):
        rel, cls, _ = items[int(k)]
        with zf.open(member_by_rel[rel]) as fh:
            img = Image.open(fh).convert("RGB")
        path = save_image(img, split, cls, i)
        records[split].append((path, cls, dhash(img)))

dump(pv_train_items, tr_idx, "lab_train")
dump(pv_train_items, va_idx, "lab_val")
dump(pv_test_items, range(len(pv_test_items)), "lab_test")
print({sp: len(records[sp]) for sp in ["lab_train", "lab_val", "lab_test"]})

## Step 3 — PlantDoc (non-lab) → `field_train` / `field_val` / `field_test`

Pemetaan kelas PlantDoc → label kanonik. Yang tidak bisa dipetakan **tidak dipakai**. Duplikat (hash gambar sama) dibuang sebelum split; gambar yang muncul di dua kelas berbeda dibuang seluruhnya.

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

pdoc = load_dataset("Project-AgML/plant_doc_classification", split="train")
pdoc_names = pdoc.features["label"].names
print("Kelas PlantDoc:", len(pdoc_names))

PLANTDOC_TO_CANON = {
    "Tomato Early blight leaf": "Tomato___Early_blight",
    "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Tomato leaf": "Tomato___healthy",
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Tomato leaf late blight": "Tomato___Late_blight",
    "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato mold leaf": "Tomato___Leaf_Mold",
    # "Tomato two spotted spider mites leaf": hanya 2 gambar -> dikecualikan (terlalu sedikit untuk di-split)
    # Target Spot: tidak ada di PlantDoc
}
missing = [k for k in PLANTDOC_TO_CANON if k not in pdoc_names]
assert not missing, f"nama kelas PlantDoc tidak ditemukan (nama berubah?): {missing}. Daftar: {pdoc_names}"
pdoc_idx_to_canon = {pdoc_names.index(k): v for k, v in PLANTDOC_TO_CANON.items()}

# kumpulkan gambar tomat PlantDoc (di memori; jumlahnya kecil, <1000)
pd_items = []
for ex in pdoc.filter(lambda lab: lab in pdoc_idx_to_canon, input_columns=["label"]):
    img = ex["image"].convert("RGB")
    pd_items.append((img, pdoc_idx_to_canon[ex["label"]], dhash(img)))
print("PlantDoc tomat (dipetakan):", len(pd_items))

# buang duplikat
by_hash = {}
for k, (_, cls, h) in enumerate(pd_items):
    by_hash.setdefault(h, []).append(k)
keep = []
dropped_dup = dropped_conflict = 0
for h, ks in by_hash.items():
    classes = {pd_items[k][1] for k in ks}
    if len(classes) > 1:
        dropped_conflict += len(ks)
    else:
        keep.append(ks[0]); dropped_dup += len(ks) - 1
print(f"duplikat dibuang: {dropped_dup}; gambar berlabel ganda dibuang: {dropped_conflict}; tersisa: {len(keep)}")

labels_keep = [pd_items[k][1] for k in keep]
# kelas dengan <4 gambar tidak bisa di-stratify 3 arah -> dikecualikan dengan peringatan
from collections import Counter
cnt = Counter(labels_keep)
too_small = {c for c, n in cnt.items() if n < 4}
if too_small:
    print("PERINGATAN kelas dikecualikan (terlalu sedikit):", too_small)
keep = [k for k in keep if pd_items[k][1] not in too_small]
labels_keep = [pd_items[k][1] for k in keep]

k_trval, k_test = train_test_split(keep, test_size=0.30, stratify=labels_keep, random_state=SEED)
k_train, k_val = train_test_split(k_trval, test_size=0.26, stratify=[pd_items[k][1] for k in k_trval], random_state=SEED)

for split, ks in [("field_train", k_train), ("field_val", k_val), ("field_test", k_test)]:
    for i, k in enumerate(ks):
        img, cls, h = pd_items[k]
        path = save_image(img, split, cls, i)
        records[split].append((path, cls, h))
print({sp: len(records[sp]) for sp in ["field_train", "field_val", "field_test"]})

## Step 4 — Anti-kebocoran: buang near-duplicate lintas set

Hash perseptual (dHash 64-bit). Gambar di set yang lebih "bersih" menjadi acuan: `field_test` dan `field_val` tidak boleh punya near-duplicate (jarak Hamming ≤ 3) di set latih.

In [ ]:
def hashes_of(split):
    return np.array([h for _, _, h in records[split]], dtype=np.uint64)

def popcount64(a):
    return np.unpackbits(np.ascontiguousarray(a).view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1)

def purge(from_splits, against_split, thr=3):
    ref = hashes_of(against_split)
    removed = 0
    for sp in from_splits:
        kept = []
        for rec in records[sp]:
            h = np.uint64(rec[2])
            if len(ref) and (popcount64(ref ^ h).min() <= thr):
                os.remove(rec[0]); removed += 1
            else:
                kept.append(rec)
        records[sp] = kept
    return removed

r1 = purge(["lab_train", "lab_val", "lab_test", "field_train", "field_val"], "field_test")
r2 = purge(["lab_train", "field_train"], "field_val")
print("near-duplicate dibuang terhadap field_test:", r1, "| terhadap field_val:", r2)

## Step 5 — Audit dataset (jumlah per kelas per set)

In [ ]:
import pandas as pd

rows = []
for c in TOMATO_CLASSES:
    rows.append({"kelas": c.replace("Tomato___", "")} | {sp: sum(1 for r in records[sp] if r[1] == c) for sp in SPLITS})
audit = pd.DataFrame(rows).set_index("kelas")
audit.loc["TOTAL"] = audit.sum()
display(audit)
print("Kelas tanpa data lapangan (uji lapangan tidak mencakupnya):",
      [c for c in TOMATO_CLASSES if not any(r[1] == c for r in records["field_test"])])

## Step 6 — Pipeline data + augmentasi domain

- Latih = campuran contoh **per-contoh**: 75% `lab_train`, 25% `field_train` (oversample data lapangan yang kecil).
- Augmentasi lebih kuat dari v1 (zoom, geser, rotasi, kecerahan/kontras, derau) agar model tidak bergantung pada latar seragam PlantVillage. Augmentasi berjalan di GPU (di dalam model latih) dan gambar di-cache sebagai uint8 — jauh lebih cepat daripada augmentasi lewat map di CPU Colab.
- Label bertipe integer (`sparse`), sehingga `class_weight` bekerja.

In [ ]:
BATCH = 32
FIELD_WEIGHT = 0.25
AUTOTUNE = tf.data.AUTOTUNE

class RandomPixelNoise(tf.keras.layers.Layer):
    # derau piksel (stddev dari skala 0-255); aktif hanya saat training
    def __init__(self, stddev=8.0, **kw):
        super().__init__(**kw)
        self.stddev = stddev
    def call(self, x, training=False):
        if not training:
            return x
        return tf.clip_by_value(x + tf.random.normal(tf.shape(x), stddev=self.stddev), 0.0, 255.0)
    def get_config(self):
        return {**super().get_config(), "stddev": self.stddev}

def make_augment():
    # dijalankan DI DALAM model (GPU); versi map() di tf.data (CPU) ~10x lebih lambat di Colab
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.15),
        tf.keras.layers.RandomZoom(0.25),
        tf.keras.layers.RandomTranslation(0.1, 0.1),
        tf.keras.layers.RandomBrightness(0.25, value_range=(0, 255)),
        tf.keras.layers.RandomContrast(0.25),
        RandomPixelNoise(8.0),
    ], name="augment")

def load_dir(split, img_size, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        f"{DATA_DIR}/{split}", class_names=TOMATO_CLASSES, label_mode="int",
        image_size=img_size, batch_size=None, shuffle=shuffle, seed=SEED,
    )

def to_u8(x, y):
    return tf.cast(x, tf.uint8), y

def to_f32(x, y):
    return tf.cast(x, tf.float32), y

def make_train_ds(img_size):
    # gambar di-cache sebagai uint8 (hemat RAM); decode JPEG hanya sekali (epoch pertama)
    lab = load_dir("lab_train", img_size, True).map(to_u8).cache().shuffle(2000, seed=SEED).repeat()
    fld = load_dir("field_train", img_size, True).map(to_u8).cache().shuffle(1000, seed=SEED).repeat()
    mixed = tf.data.Dataset.sample_from_datasets([lab, fld], weights=[1 - FIELD_WEIGHT, FIELD_WEIGHT], seed=SEED)
    return mixed.batch(BATCH).map(to_f32, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

def make_eval_ds(split, img_size):
    return load_dir(split, img_size, False).map(to_u8).cache().map(to_f32).batch(BATCH).prefetch(AUTOTUNE)

def make_val_ds(img_size):
    ds = load_dir("lab_val", img_size, False).concatenate(load_dir("field_val", img_size, False))
    return ds.map(to_u8).cache().map(to_f32).batch(BATCH).prefetch(AUTOTUNE)

n_lab, n_fld = len(records["lab_train"]), len(records["field_train"])
STEPS_PER_EPOCH = (n_lab + n_fld) // BATCH
print("lab_train:", n_lab, "| field_train:", n_fld, "| steps/epoch:", STEPS_PER_EPOCH)

# class weight dari gabungan jumlah latih (efektif membantu kelas langka)
counts = np.array([sum(1 for r in records["lab_train"] + records["field_train"] if r[1] == c) for c in TOMATO_CLASSES], dtype=float)
CLASS_WEIGHT = {i: float(counts.sum() / (len(counts) * max(n, 1))) for i, n in enumerate(counts)}
print("class_weight:", {TOMATO_CLASSES[i].replace('Tomato___', ''): round(w, 2) for i, w in CLASS_WEIGHT.items()})

## Step 7 — Model builder & training (dua kandidat)

Preprocessing dibakar ke dalam graf model, sehingga input inferensi = piksel mentah 0–255 (kontrak `ml-service/app/ml/inference.py`; **jangan dibagi 255**):
- MobileNetV2 → `preprocess_input` menskala ke [-1, 1].
- EfficientNetB0 → normalisasi sudah built-in, `preprocess_input` pass-through.

Kandidat bisa diubah di `CANDIDATES`. Dua tahap seperti v1: basis beku (Adam 1e-3), lalu 30 lapisan terakhir dibuka (Adam 1e-5).

In [ ]:
CANDIDATES = {
    "mobilenet_v2": (224, 224),
    "efficientnet_b0": (224, 224),
}

def build_model(arch, num_classes, img_size):
    input_shape = tuple(img_size) + (3,)
    inputs = tf.keras.Input(shape=input_shape)
    if arch == "mobilenet_v2":
        preprocess = tf.keras.applications.mobilenet_v2.preprocess_input
        base = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights="imagenet")
    elif arch == "efficientnet_b0":
        preprocess = tf.keras.applications.efficientnet.preprocess_input
        base = tf.keras.applications.EfficientNetB0(input_shape=input_shape, include_top=False, weights="imagenet")
    else:
        raise ValueError(arch)
    base.trainable = False
    x = preprocess(inputs)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs), base

def compile_model(model, lr):
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

def train_candidate(arch, img_size):
    train_ds, val_ds = make_train_ds(img_size), make_val_ds(img_size)
    core, base = build_model(arch, len(TOMATO_CLASSES), img_size)
    model = tf.keras.Sequential([tf.keras.Input(shape=tuple(img_size) + (3,)), make_augment(), core], name="train_wrapper")

    cb = lambda p: [tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=p, restore_best_weights=True)]

    compile_model(model, 1e-3)
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=12, steps_per_epoch=STEPS_PER_EPOCH,
                   class_weight=CLASS_WEIGHT, callbacks=cb(3))

    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    compile_model(model, 1e-5)
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=10, steps_per_epoch=STEPS_PER_EPOCH,
                   class_weight=CLASS_WEIGHT, callbacks=cb(4))
    return core, (h1.history, h2.history)   # core = model tanpa lapisan augmentasi; dipakai untuk evaluasi dan ekspor

trained = {}
for arch, size in CANDIDATES.items():
    print(f"\n########## Melatih {arch} {size} ##########")
    trained[arch] = {"size": size, "model": None}
    trained[arch]["model"], trained[arch]["history"] = train_candidate(arch, size)

## Step 8 — Evaluasi (lab vs lapangan, terpisah) + pilih model

Metrik: akurasi (+ selang Wilson 95%), presisi/recall/F1 per kelas dan rata-rata makro. Pada `field_test`, rata-rata makro hanya dihitung untuk kelas yang punya data (Target Spot dan spider mites tidak punya data lapangan). Selang kepercayaan mengasumsikan gambar independen — di PlantDoc itu belum tentu benar, jadi baca sebagai perkiraan kasar.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

OUT = "/content/outputs"
os.makedirs(OUT, exist_ok=True)

def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (centre - half, centre + half)

def collect(predict_fn, ds):
    y_true, y_pred = [], []
    for x, y in ds:
        probs = predict_fn(x.numpy())
        y_true.extend(y.numpy().tolist()); y_pred.extend(np.argmax(probs, axis=1).tolist())
    return np.array(y_true), np.array(y_pred)

def evaluate(predict_fn, ds, title, tag):
    y_true, y_pred = collect(predict_fn, ds)
    n = len(y_true); k = int((y_true == y_pred).sum())
    present = sorted(set(y_true.tolist()))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=list(range(len(TOMATO_CLASSES))), zero_division=0)
    lo, hi = wilson(k, n)
    res = {
        "title": title, "n": n, "accuracy": k / n, "ci95": [lo, hi],
        "macro_precision": float(np.mean([p[i] for i in present])),
        "macro_recall": float(np.mean([r[i] for i in present])),
        "macro_f1": float(np.mean([f[i] for i in present])),
        "classes_evaluated": [TOMATO_CLASSES[i] for i in present],
        "per_class": {TOMATO_CLASSES[i]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]), "n": int(s[i])} for i in range(len(TOMATO_CLASSES))},
    }
    print(f"=== {title} === n={n} akurasi={res['accuracy']*100:.2f}% CI95=[{lo*100:.1f}, {hi*100:.1f}] macro-F1={res['macro_f1']*100:.2f}%")
    for i in range(len(TOMATO_CLASSES)):
        if s[i] > 0:
            print(f"  {TOMATO_CLASSES[i].replace('Tomato___',''):45s} P={p[i]*100:5.1f} R={r[i]*100:5.1f} F1={f[i]*100:5.1f} n={int(s[i])}")
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(TOMATO_CLASSES))))
    short = [c.replace("Tomato___", "") for c in TOMATO_CLASSES]
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=short, yticklabels=short, cmap="Greens")
    plt.title(f"Confusion Matrix — {title}"); plt.ylabel("Aktual"); plt.xlabel("Prediksi")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout()
    plt.savefig(f"{OUT}/confusion_matrix_{tag}.png", dpi=150); plt.show()
    return res

metrics = {}
for arch, d in trained.items():
    size = d["size"]; model = d["model"]
    fn = lambda x, m=model: m.predict(x, verbose=0)
    metrics[arch] = {
        "lab_val": evaluate(fn, make_eval_ds("lab_val", size), f"{arch} — lab_val", f"{arch}_lab_val"),
        "field_val": evaluate(fn, make_eval_ds("field_val", size), f"{arch} — field_val", f"{arch}_field_val"),
    }

BEST = max(metrics, key=lambda a: metrics[a]["field_val"]["accuracy"])
print("\nModel terpilih (akurasi field_val tertinggi):", BEST)
for a in metrics:
    print(f"  {a:18s} field_val={metrics[a]['field_val']['accuracy']*100:.2f}%  lab_val={metrics[a]['lab_val']['accuracy']*100:.2f}%")

In [ ]:
# UJI AKHIR — hanya model terpilih, sekali saja (jangan memilih model lagi berdasarkan hasil ini)
best_model = trained[BEST]["model"]; best_size = trained[BEST]["size"]
fn_best = lambda x: best_model.predict(x, verbose=0)
final = {
    "lab_test": evaluate(fn_best, make_eval_ds("lab_test", best_size), f"v2 {BEST} — lab_test", "v2_lab_test"),
    "field_test": evaluate(fn_best, make_eval_ds("field_test", best_size), f"v2 {BEST} — field_test", "v2_field_test"),
}

In [ ]:
# BASELINE v1 (opsional): bandingkan model lama pada set uji YANG SAMA.
# Upload ml-service/models/tomato_classifier.tflite (v1) saat diminta, atau Cancel untuk melewati.
from google.colab import files
baseline = None
try:
    up = files.upload()
    if up:
        v1_path = list(up.keys())[0]
        interp = tf.lite.Interpreter(model_path=v1_path); interp.allocate_tensors()
        inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
        v1_size = tuple(int(v) for v in inp["shape"][1:3])
        def predict_v1(x):
            res = []
            for im in x:
                interp.set_tensor(inp["index"], im[None].astype(np.float32)); interp.invoke()
                res.append(interp.get_tensor(out["index"])[0])
            return np.array(res)
        baseline = {
            "lab_test": evaluate(predict_v1, make_eval_ds("lab_test", v1_size), "v1 baseline — lab_test", "v1_lab_test"),
            "field_test": evaluate(predict_v1, make_eval_ds("field_test", v1_size), "v1 baseline — field_test", "v1_field_test"),
        }
except Exception as e:
    print("Baseline dilewati:", repr(e))

print("\n===== RINGKASAN =====")
for name, res in [("v1 (baseline)", baseline), ("v2", final)]:
    if res:
        print(f"{name:14s} lab_test={res['lab_test']['accuracy']*100:.2f}%  field_test={res['field_test']['accuracy']*100:.2f}% (n={res['field_test']['n']})")

## Step 9 — Export TFLite + verifikasi + label map

TFLite diverifikasi terhadap model Keras (kesepakatan argmax) sebelum dipakai. `label_map.json` hanya memuat kunci `tomato` dan **mempertahankan urutan 10 kelas** agar `disease_reference` di database tidak perlu diubah.

In [ ]:
TFLITE_PATH = f"{OUT}/tomato_classifier.tflite"
best_model.export("/content/saved_model_v2")   # Keras 3: ekspor SavedModel; from_keras_model gagal di Keras 3 / TF 2.21
converter = tf.lite.TFLiteConverter.from_saved_model("/content/saved_model_v2")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]   # kuantisasi dinamis (int8) merusak EfficientNetB0: argmax Keras vs TFLite hanya 92,7%; float16 = 100%
tflite_bytes = converter.convert()
open(TFLITE_PATH, "wb").write(tflite_bytes)
print(f"Tersimpan: {TFLITE_PATH} ({len(tflite_bytes)/1e6:.2f} MB)  [v1 = 2.5 MB; Railway: pertahankan di bawah ~20 MB]")

# verifikasi TFLite vs Keras pada 300 gambar (campuran lab_test + field_test)
interp2 = tf.lite.Interpreter(model_path=TFLITE_PATH); interp2.allocate_tensors()
i2, o2 = interp2.get_input_details()[0], interp2.get_output_details()[0]
assert tuple(i2["shape"][1:3]) == tuple(best_size), "ukuran input TFLite tidak sama dengan best_size"
xs = []
for split in ["lab_test", "field_test"]:
    # shuffle=True supaya 150 gambar mencakup banyak kelas (urutan direktori = per kelas)
    for x, y in load_dir(split, best_size, True).take(150):
        xs.append(x.numpy())
xs = np.stack(xs)
keras_probs = best_model.predict(xs, verbose=0)
tfl_probs = []
for im in xs:
    interp2.set_tensor(i2["index"], im[None].astype(np.float32)); interp2.invoke()
    tfl_probs.append(interp2.get_tensor(o2["index"])[0])
tfl_probs = np.array(tfl_probs)
agree = float((keras_probs.argmax(1) == tfl_probs.argmax(1)).mean())
print(f"Kesepakatan argmax Keras vs TFLite: {agree*100:.1f}%  | selisih prob maks: {np.abs(keras_probs - tfl_probs).max():.4f}")
assert agree >= 0.97, "TFLite menyimpang dari Keras — jangan deploy, periksa konversi/kuantisasi"

In [ ]:
def slugify(name):
    s = re.sub(r"[^a-zA-Z0-9]+", "_", name.strip())
    return s.strip("_").lower()

def to_disease_label(crop, raw_class_name):
    s = slugify(raw_class_name)
    if s.startswith("healthy"):
        return f"{crop}_healthy"
    if s.startswith(crop + "_"):
        return s
    return f"{crop}_{s}"

label_map = {
    "tomato": {
        "model_version": "v2",
        "img_size": list(best_size),
        "source_dataset": "huggingface:mohanty/PlantVillage (color, split per daun) + huggingface:Project-AgML/plant_doc_classification (subset tomat, split stratified)",
        "architecture": BEST,
        "classes": [
            {"index": i, "raw_class_name": name, "disease_label": to_disease_label("tomato", name)}
            for i, name in enumerate(TOMATO_CLASSES)
        ],
    }
}
json.dump(label_map, open(f"{OUT}/label_map.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)

# bandingkan dengan 10 disease_label yang ada di database/schema.sql (urutan & ejaan harus sama dengan v1)
EXPECTED = ["tomato_bacterial_spot", "tomato_early_blight", "tomato_late_blight", "tomato_leaf_mold",
            "tomato_septoria_leaf_spot", "tomato_spider_mites_two_spotted_spider_mite", "tomato_target_spot",
            "tomato_tomato_yellow_leaf_curl_virus", "tomato_tomato_mosaic_virus", "tomato_healthy"]
got = [c["disease_label"] for c in label_map["tomato"]["classes"]]
assert got == EXPECTED, f"disease_label berubah dari v1! got={got}"
print("OK — 10 disease_label identik dengan v1 / schema.sql")

metrics_out = {
    "selected_architecture": BEST,
    "img_size": list(best_size),
    "tflite_mb": len(tflite_bytes) / 1e6,
    "tflite_keras_argmax_agreement": agree,
    "candidates_validation": metrics,
    "v2_final_test": final,
    "v1_baseline_test": baseline,
    "data_counts": audit.to_dict(),
    "notes": [
        "lab_test dan lab_val dipisah per leaf_id (mohanty/PlantVillage).",
        "field_* dari PlantDoc subset tomat; Target Spot dan spider mites tidak punya data lapangan.",
        "Model dipilih berdasar akurasi field_val; uji akhir dijalankan sekali pada model terpilih.",
        "Selang Wilson mengasumsikan gambar independen; PlantDoc bisa punya near-duplicate lolos filter.",
    ],
}
json.dump(metrics_out, open(f"{OUT}/metrics.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False, default=float)
audit.to_csv(f"{OUT}/data_audit.csv")
print("metrics.json, label_map.json, data_audit.csv tersimpan di", OUT)

In [ ]:
# Bundel hasil untuk didownload
!cd /content/outputs && zip -q -r /content/agrivision_tomato_v2.zip .
files.download("/content/agrivision_tomato_v2.zip")

## Step 10 — Setelah download

1. Extract `agrivision_tomato_v2.zip`.
2. Jangan langsung menimpa model produksi: serahkan `tomato_classifier.tflite`, `label_map.json`, dan `metrics.json` ke sesi Claude Code proyek ini. Integrasi (ganti file di `ml-service/models/`, `pytest`, push, cek `/api/health` melaporkan `model_version = v2`) dikerjakan di sana.
3. Simpan `confusion_matrix_v2_lab_test.png` dan `confusion_matrix_v2_field_test.png` untuk laporan; laporkan **lab dan lapangan terpisah** beserta catatan keterbatasan (tidak ada uji Target Spot/spider mites di lapangan, `field_test` kecil, selang kepercayaan lebar).
4. Kalau akurasi lapangan v2 tidak lebih baik dari v1 pada `field_test`, **jangan deploy** — laporkan apa adanya.